# Tutorial 4: integral families, IBP and master integrals

**What you will learn**
1. what an integral family is and how to write one
2. sectors, and which of them are zero
3. the IBP identities themselves
4. reduction to master integrals with `ibp_reduce`
5. how to count the masters without any reduction (Lee and Pomeransky)
6. a two-loop number from the masters alone: the kite is $6\zeta(3)/q^2$

The ideas are explained in the lecture note, Sections 8, 18 and 19 (master integrals). Here we do them on the computer.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))     # use the feynsage of this repository (not needed after ./install.sh)
from feynsage import *
%display latex

## Step 1: a family

A family is a list of propagators $D_1, \dots, D_t$. Its members are
$$F(a_1, \dots, a_t) = \int \frac{d^d\ell}{D_1^{a_1}\cdots D_t^{a_t}}, \qquad a_i \in \mathbb Z .$$
A positive $a_i$ is a propagator ($a_i - 1$ dots), a negative one is a numerator. We take the equal-mass bubble of the note in Euclidean space with $m = 1$. A propagator with a mass is written `("momentum", "mass")`.

In [2]:
bub = family([("l", "1"), ("l - p", "1")], kin={"p^2": "pp"}, euclidean=True)
bub.info()

Integral family 'F'
  loop momenta: l   external momenta: p   Euclidean metric
  D1 = (l)^2 + 1
  D2 = (l - 1*p)^2 + 1
  An integral F(a1,...,at) is  Int prod_r d^d l_r  prod_i D_i^(-a_i); a_i <= 0 are numerators.


## Step 2: sectors

The *sector* of $F(a_1, a_2)$ is the set of lines that are present ($a_i > 0$). The bubble has four sectors: (1,1), (1,0), (0,1), (0,0). `is_zero_sector` tells which ones vanish. It uses Lee's criterion: a sector is zero if its integrals are scaleless.

In [3]:
for sec in [(1, 1), (1, 0), (0, 1), (0, 0)]:
    print(sec, "zero" if bub.is_zero_sector(sec) else "non-zero")

(1, 1) non-zero
(1, 0) non-zero
(0, 1) non-zero
(0, 0) zero


If the lines are massless, the one-line sectors are massless tadpoles and they vanish:

In [4]:
bub0 = family(["l", "l - p"], kin={"p^2": "pp"}, euclidean=True)
[(sec, bub0.is_zero_sector(sec)) for sec in [(1, 1), (1, 0), (0, 1)]]

[((1, 1), False), ((1, 0), True), ((0, 1), True)]

## Step 3: the IBP identities

`ibp(a)` gives the identities for the seed $F(a)$: one for each vector ($\ell$ and $p$) put inside the derivative $\partial/\partial\ell^\mu$. Each identity is a dictionary {index: coefficient} meaning $\sum c\,F(\text{index}) = 0$, with $d$ the dimension.

In [5]:
ids = bub.ibp((1, 1))
for k, ident in enumerate(ids):
    print("identity %d:" % (k + 1), " + ".join("(%s) F%s" % (c, a) for a, c in ident.items()), "= 0")

identity 1: (d - 3) F(1, 1) + (2) F(2, 1) + (pp + 2) F(1, 2) + (-1) F(0, 2) = 0
identity 2: (-pp) F(2, 1) + (1) F(2, 0) + (pp) F(1, 2) + (-1) F(0, 2) = 0


The first one is the identity derived by hand in the note (Section 8),
$$0 = (D - 3)\,J(1,1) + (p^2 + 4m^2)\,J(2,1) - T(2) ,$$
once the symmetry $F(1,2) = F(2,1)$ and $F(0,2) = T(2)$ are used.

## Step 4: reduce to master integrals

`ibp_reduce(family, targets)` chooses the seeds, finds the symmetries of the family by itself and solves the system.

In [6]:
red = ibp_reduce(bub, ["F(2,1)", "F(2,2)", "F(3,1)"])
red

F(2, 1) = [(-1) * (pp + 4)^-1 * (d - 3)] F(1, 1) + [(-1/2) * (pp + 4)^-1 * (d - 2)] F(0, 1)
F(2, 2) = [(-1) * (pp + 4)^-2 * pp^-1 * (d - 2) * (2*d + pp - 6)] F(0, 1) + [(pp + 4)^-2 * pp^-1 * (d - 3) * (d*pp - 6*pp - 4)] F(1, 1)
F(3, 1) = [(1/8) * (pp + 4)^-2 * pp^-1 * (d - 2) * (d*pp^2 + 8*d*pp - 4*pp^2 + 8*d - 32*pp - 24)] F(0, 1) + [(1/2) * (pp + 4)^-2 * pp^-1 * (d - 3) * (d*pp - 4*pp + 4)] F(1, 1)

In [7]:
red.info()
print("masters:", red.masters)

Reduction of 3 integrals of family 'F' with the ff reducer (0.1 s).
Each target F(a) is written as sum_M c_M(d, invariants) M over the master integrals
M = F(0, 1), F(1, 1); d is the space-time dimension.
masters: [(0, 1), (1, 1)]


**Check against the note.** The note (eq. J21, Euclidean) has
$$J(2,1) = -\frac{D - 3}{p^2 + 4m^2}\,J(1,1) + \frac{1}{p^2 + 4m^2}\,T(2), \qquad T(2) = -\frac{D - 2}{2m^2}\,T(1) .$$
With $m = 1$ the coefficient of $T(1) = F(0,1)$ is $-(D-2)/(2(p^2 + 4))$. Compare exactly:

In [8]:
row = red[(2, 1)]
dd, ppp = row[(1, 1)].parent().gens()          # the symbols d and pp of the coefficients
print(row[(1, 1)] == -(dd - 3)/(ppp + 4), row[(0, 1)] == -(dd - 2)/(2*(ppp + 4)))

True True


## Step 5: the masters list shows what the targets need

`red.masters` lists only the masters that appear in the answers. The massless box has three masters, but $F(2,1,1,1)$ needs only two of them. Here $s$ is a symbol and $t$ is set to 1 (a single scale besides $d$ keeps the finite-field reducer fast).

In [9]:
box = family(["k", "k + p1", "k + p1 + p2", "k - p3"],
             kin={"p1^2": 0, "p2^2": 0, "p3^2": 0, "p1.p2": "s/2", "p1.p3": "1/2", "p2.p3": "-(s+1)/2"})
print(ibp_reduce(box, ["F(2,1,1,1)"]).masters)
print(ibp_reduce(box, ["F(2,1,1,1)", "F(1,2,1,1)"]).masters)

[(0, 1, 0, 1), (1, 1, 1, 1)]


[(0, 1, 0, 1), (1, 0, 1, 0), (1, 1, 1, 1)]


The three are the box $F(1,1,1,1)$, the $s$-channel bubble $F(1,0,1,0)$ and the $t$-channel bubble $F(0,1,0,1)$.

## Step 6: count the masters without reducing anything

Lee and Pomeransky (2013): in each sector, set the $x_i$ of the missing lines to zero, form $G = \mathcal U + \mathcal F$ and count the points where all derivatives of $G$ vanish with every remaining $x_i \ne 0$. That number is the number of masters of the sector. (Section 19 of the note does the bubble by hand.)

The function below does it with a Gröbner basis. The extra variable $t$ with $t\prod x_i = 1$ removes the solutions with some $x_i = 0$. Numbers are used for the kinematics.

In [10]:
def critical_points(fam, sector, numbers):
    U, F = fam.UF()
    n = len(sector)
    R = PolynomialRing(QQ, ['x%d' % (i + 1) for i in range(n)] + ['t'])
    x, t = R.gens()[:n], R.gens()[n]
    G = R(str(SR(str(U + F)).subs(numbers)))
    live = [x[i] for i in range(n) if sector[i]]
    I = R.ideal([G.derivative(v) for v in live] + [x[i] for i in range(n) if not sector[i]] + [t*prod(live) - 1])
    return 0 if I.is_one() else I.vector_space_dimension()

pp = var('pp')
for sec in [(1, 1), (1, 0), (0, 1)]:
    print(sec, critical_points(bub, sec, {pp: 7/3}))

(1, 1) 1
(1, 0) 1
(0, 1) 1


Top sector 1, each tadpole sector 1. The two tadpole sectors are the same integral (shift $\ell \to \ell + p$), so it is counted once: $1 + 1 = 2$ masters, as IBP found. For the box, add up all non-empty sectors:

In [11]:
from itertools import product as cartesian
s = var('s')
total = 0
for sec in cartesian([0, 1], repeat=4):
    if sum(sec) == 0:
        continue
    c = critical_points(box, sec, {s: -11/5})
    total += c
    if c:
        print(sec, c)
print("total:", total)

(0, 1, 0, 1) 1
(1, 0, 1, 0) 1
(1, 1, 1, 1) 1
total: 3


Three masters: the box and the two bubbles. Every triangle and every one-line sector gives zero.

## Step 7: a two-loop number from two masters

The massless two-loop propagator family of the note (Section 18) with $q^2 = 1$, Euclidean:

In [12]:
kite = family(["l1", "l1 + q", "l1 + l2", "l1 + l2 + q", "l2"], kin={"q^2": 1}, euclidean=True)
kite.info()
print("symmetries:", symmetries(kite))
rk = ibp_reduce(kite, ["F(1,1,1,1,1)"])
rk

Integral family 'F'
  loop momenta: l1, l2   external momenta: q   Euclidean metric
  D1 = (l1)^2
  D2 = (l1 + q)^2
  D3 = (l1 + l2)^2
  D4 = (l1 + l2 + q)^2
  D5 = (l2)^2
  An integral F(a1,...,at) is  Int prod_r d^d l_r  prod_i D_i^(-a_i); a_i <= 0 are numerators.
symmetries: [(1, 0, 3, 2, 4), (2, 3, 0, 1, 4)]


F(1, 1, 1, 1, 1) = [(-2) * (d - 4)^-1 * (d - 3)] F(1, 1, 1, 1, 0) + [(18) * (d - 4)^-2 * (d - 10/3) * (d - 8/3)] F(0, 1, 1, 0, 1)

The kite itself is *not* a master. It is a combination of the sunset $F(0,1,1,0,1)$ and the product of two bubbles $F(1,1,1,1,0)$. Both are Gamma functions. With the one-loop function $G(a,b)$ of the massless bubble,
- bubble × bubble $= G(1,1)^2$
- sunset: do the $\ell_2$ bubble first, it gives $G(1,1)\,(\ell_1^2)^{D/2-2}$. The $\ell_1$ integral is then a bubble with powers 1 and $2 - D/2$: sunset $= G(1,1)\,G(1, 2 - D/2)$.

In [13]:
from feynsage.oneloop import G, D, expand_eps
master_value = {(0, 1, 1, 0, 1): G(1, 1)*G(1, 2 - D/2), (1, 1, 1, 1, 0): G(1, 1)^2}
dvar = SR.var('d')
kite_value = sum(SR(str(c)).subs({dvar: D}) * master_value[mm] for mm, c in rk[(1, 1, 1, 1, 1)].items())
expand_eps(kite_value, 0, loops=2)

6*zeta(3)

All the poles and all the rational numbers cancel between the two terms and exactly $6\zeta(3)$ is left. This is the value of the note, found there with Feynman parameters and Cheng-Wu. Here it came from pure algebra plus two Gamma-function integrals.

## Step 8: three reducers, one answer

`method="ff"` (finite fields, like Kira and FIRE), `"trimmed"` (exact arithmetic on only the equations the targets need) and `"exact"` (exact arithmetic on everything) give the same coefficients.

In [14]:
res = {}
for meth in ["ff", "trimmed", "exact"]:
    res[meth] = ibp_reduce(kite, ["F(2,2,1,2,2)"], method=meth)
    print("%-8s %.2f s" % (meth, res[meth].seconds))
print("all equal:", res["ff"].table == res["trimmed"].table == res["exact"].table)

ff       0.11 s
trimmed  0.10 s


exact    4.65 s
all equal: True


## Exercises

**1.** The bubble with two *different* masses has three masters. Find them with `ibp_reduce` and check the number with `critical_points`.
<details><summary>Show a solution</summary>

```python
b2 = family([("l", "1"), ("l - p", "2")], kin={"p^2": "pp"}, euclidean=True)
print(ibp_reduce(b2, ["F(2,1)", "F(1,2)"]).masters)
pp = var('pp')
print(sum(critical_points(b2, sec, {pp: 7/3}) for sec in [(1, 1), (1, 0), (0, 1)]))   # the tadpoles differ now
```
</details>

**2.** Reduce a kite integral with a numerator, for example $F(1,1,1,1,-1)$ (the line $D_5$ is upstairs). Does a new master appear?
<details><summary>Show a solution</summary>

```python
ibp_reduce(kite, ["F(1,1,1,1,-1)"])
```
</details>

**3.** In the kite reduction the coefficient of the sunset has $(d-4)^2$ in the denominator. To get the kite to order $\epsilon^0$, how many orders in $\epsilon$ of the sunset do you need? Check by expanding `master_value` yourself.
<details><summary>Show a solution</summary>

```python
# the coefficient is ~ 1/eps^2, so the sunset is needed to order eps^2 (two orders beyond eps^0)
expand_eps(master_value[(0, 1, 1, 0, 1)], 2, loops=2)
```
</details>